# 11. Compositional and expression shifts with UMAP LDA z-scores

This notebook quantifies how knockout alters the GABAergic landscape at three levels:

1. **Compositional shifts**  
   - CLR-transformed cluster proportions per sample  
   - Welch t-tests (KO vs WT) with Benjamini–Hochberg FDR

2. **Expression shifts**  
   - Energy distance (e-distance) between WT and KO in PCA space  
   - Observed statistic from `pertpy`  
   - Permutation p-values from a fast NumPy implementation

3. **Cell-level WT<-->KO axis**  
   - Linear Discriminant Analysis (LDA) on PCA coordinates  
   - LDA z-score per cell (“how KO-like is this cell?”)  
   - UMAP colored by LDA z-score, split by condition

Cluster 14 and the outlier sample `96_6_M_KO` are excluded from all analyses.

## 11.1 Imports and settings

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pertpy as pt
from scipy import stats
from scipy.spatial.distance import cdist
from statsmodels.stats.multitest import multipletests
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Scanpy and plotting defaults
sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)
sns.set(style="whitegrid")

# Column keys
cluster_key   = "leiden_0.7"
condition_key = "condition"
sample_key    = "batch"

# Reference / test groups
ref_group  = "WT"
test_group = "KO"

# Filtering parameters
REMOVE_CLUSTER   = "14"        # fully excluded everywhere
REMOVE_SAMPLE    = "96_6_M_KO" # outlier: aberrant composition
MIN_CELLS_SAMPLE = 200
N_PERMS          = 999
MAX_CELLS_PERM   = 400
ALPHA            = 0.05

# Cluster name mapping (cluster 14 intentionally absent)
cluster_names = {
    "0":  "mge_prog_prdm16",
    "1":  "cge_prog_nr2f2",
    "2":  "mge_neu_sox6",
    "3":  "lge_neu_ddah1",
    "4":  "mge_neu_st18",
    "5":  "lge_prog_ebf1",
    "6":  "lge_prog_isl1",
    "7":  "mge_prog_lhx6",
    "8":  "mge_neu_erbb4",
    "9":  "mge_prog_sp9",
    "10": "cge_prog_sp8",
    "11": "cge_prog_pax6",
    "12": "mge_prog_nkx21",
    "13": "cge_neu_synpr",
}
cluster_order = [cluster_names[str(i)] for i in range(14) if str(i) in cluster_names]


def remove_spines(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def energy_distance_np(X, Y):
    """
    Numpy energy distance for permutation testing.
    E(X,Y) = 2·mean||x-y|| − mean||x-x'|| − mean||y-y'||
    """
    xy = cdist(X, Y).mean()
    xx = cdist(X, X).mean()
    yy = cdist(Y, Y).mean()
    return 2 * xy - xx - yy

## 11.2 Load GABA AnnData and apply filters

We load the GABA-reclustered object, standardize metadata, and:

- remove cluster 14 entirely,
- drop the outlier sample `96_6_M_KO`,
- restrict to samples with at least 200 cells,
- map numeric clusters to descriptive names.

In [ ]:
adata = sc.read_h5ad("gaba_reclustered.h5ad")

adata.obs[cluster_key]   = adata.obs[cluster_key].astype(str)
adata.obs[condition_key] = adata.obs[condition_key].astype(str).str.strip().str.upper()
adata.obs[sample_key]    = adata.obs[sample_key].astype(str)

# Remove cluster 14
adata = adata[adata.obs[cluster_key] != REMOVE_CLUSTER].copy()

# Remove outlier sample and small samples
sample_counts = adata.obs[sample_key].value_counts()
keep_samples = [
    s for s in sample_counts[sample_counts >= MIN_CELLS_SAMPLE].index
    if s != REMOVE_SAMPLE
]
adata = adata[adata.obs[sample_key].isin(keep_samples)].copy()

# Map cluster IDs to names
adata.obs["cluster_name"] = (
    adata.obs[cluster_key].map(cluster_names)
    .fillna(adata.obs[cluster_key]).astype(str)
)
adata.obs["cluster_name"] = pd.Categorical(
    adata.obs["cluster_name"], categories=cluster_order, ordered=True
)

print(f"Samples kept : {sorted(keep_samples)}")
print(f"Total cells  : {adata.n_obs}  (cluster 14 removed)")
print("Conditions:")
print(adata.obs[condition_key].value_counts())
print(f"Clusters     : {cluster_order}")

## 11.3 Plot 1 – Compositional shifts (CLR + Welch t-test)

We compute per-sample cluster proportions, apply a centered log-ratio (CLR) transform, and test KO vs WT for each cluster:

- Welch t-test on CLR values,
- Benjamini–Hochberg FDR correction,
- horizontal bar plot with 95% confidence intervals.

In [ ]:
# Sample metadata and counts
sample_meta = (
    adata.obs[[sample_key, condition_key]]
    .drop_duplicates()
    .set_index(sample_key)
)
counts_mat = pd.crosstab(adata.obs[sample_key], adata.obs["cluster_name"])

# Proportions and CLR
props = counts_mat.div(counts_mat.sum(axis=1), axis=0)
eps = 0.5 / counts_mat.sum(axis=1).mean()
log_props = np.log(props + eps)
clr = log_props.sub(log_props.mean(axis=1), axis=0)
clr[condition_key] = sample_meta.loc[clr.index, condition_key]

# Per-cluster Welch t-tests
comp_rows = []
for cl in cluster_order:
    wt = clr.loc[clr[condition_key] == ref_group, cl].values
    ko = clr.loc[clr[condition_key] == test_group, cl].values
    if len(wt) < 2 or len(ko) < 2:
        continue

    _, p = stats.ttest_ind(ko, wt, equal_var=False)
    effect = ko.mean() - wt.mean()
    se = np.sqrt(wt.var(ddof=1) / len(wt) + ko.var(ddof=1) / len(ko))
    ci_h = stats.t.ppf(0.975, len(wt) + len(ko) - 2) * se

    comp_rows.append(
        {
            "cluster_name": cl,
            "effect_CLR": effect,
            "CI_lo": effect - ci_h,
            "CI_hi": effect + ci_h,
            "pvalue": p,
        }
    )

comp_df = pd.DataFrame(comp_rows)
_, padj, _, _ = multipletests(comp_df["pvalue"], method="fdr_bh")
comp_df["padj"] = padj
comp_df["significant"] = comp_df["padj"] < ALPHA
comp_df = comp_df.sort_values("effect_CLR").reset_index(drop=True)
comp_df.to_csv("compositional_clr.csv", index=False)

print(
    comp_df[
        ["cluster_name", "effect_CLR", "CI_lo", "CI_hi", "pvalue", "padj", "significant"]
    ].to_string(index=False)
)

# Plot
fig, ax = plt.subplots(figsize=(7, 8), facecolor="white")
for i, row in comp_df.iterrows():
    eff = row["effect_CLR"]
    sig = row["significant"]
    col = (
        "#D4537E" if (sig and eff > 0) else
        "#378ADD" if (sig and eff < 0) else
        "#CCCCCC"
    )

    ax.barh(i, eff, color=col, alpha=0.88, edgecolor="white", height=0.6)
    ax.plot(
        [row["CI_lo"], row["CI_hi"]],
        [i, i],
        color="black",
        linewidth=1.5,
        solid_capstyle="butt",
    )
    ax.plot(
        [row["CI_lo"], row["CI_hi"]],
        [i, i],
        color="black",
        linewidth=0,
        marker="|",
        markeredgewidth=1.5,
        markersize=7,
    )

ax.set_yticks(range(len(comp_df)))
ax.set_yticklabels(comp_df["cluster_name"].values, fontsize=9)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("CLR effect: KO − WT  (95% CI, BH padj)", fontsize=10)
ax.set_title(
    "Compositional shift (KO vs WT)\nCLR proportions · Welch t-test · BH",
    fontsize=11,
    fontweight="bold",
)

ax.legend(
    handles=[
        mpatches.Patch(color="#D4537E", label=f"Enriched in KO (padj < {ALPHA})"),
        mpatches.Patch(color="#378ADD", label=f"Depleted in KO (padj < {ALPHA})"),
        mpatches.Patch(color="#CCCCCC", label="Not significant"),
    ],
    frameon=False,
    fontsize=9,
)
remove_spines(ax)
plt.tight_layout()
plt.savefig("compositional_clr.png", dpi=300, bbox_inches="tight")
plt.show()
print("Plot 1 saved: compositional_clr.png")

## 11.4 Plot 2 – Expression shift (energy distance)

We quantify expression shifts between WT and KO:

- observed energy distance from `pertpy.tl.Distance(metric="edistance")`,
- permutation p-values using a fast NumPy implementation on PCA space,
- violin plot of per-cell distance z-scores (`edist_z`) per cluster.

In [ ]:
dist_tool = pt.tl.Distance(metric="edistance")
X_pca_all = np.asarray(adata.obsm["X_pca"])
rng = np.random.default_rng(42)

dist_rows = []
for cl in cluster_order:
    mask = (adata.obs["cluster_name"] == cl).values
    sub = adata[mask]

    n_wt = (sub.obs[condition_key] == ref_group).sum()
    n_ko = (sub.obs[condition_key] == test_group).sum()
    if n_wt < 5 or n_ko < 5:
        print(f"  Skipping {cl}: WT={n_wt}, KO={n_ko}")
        continue

    # Observed e-distance via pertpy
    dist_mat = dist_tool.pairwise(sub, groupby=condition_key)
    if ref_group in dist_mat.index:
        obs_d = float(dist_mat.loc[ref_group, test_group])
    else:
        obs_d = float(dist_mat.values[0, 1])

    # Permutation p-value using NumPy implementation
    X_cl = X_pca_all[mask]
    labels = sub.obs[condition_key].values.copy()
    n_sub = min(min(n_wt, n_ko), MAX_CELLS_PERM)

    perm_ds = []
    for _ in range(N_PERMS):
        shuf = labels.copy()
        rng.shuffle(shuf)
        Xa = X_cl[shuf == ref_group]
        Xb = X_cl[shuf == test_group]

        if len(Xa) > n_sub:
            Xa = Xa[rng.choice(len(Xa), n_sub, replace=False)]
        if len(Xb) > n_sub:
            Xb = Xb[rng.choice(len(Xb), n_sub, replace=False)]

        if len(Xa) > 1 and len(Xb) > 1:
            perm_ds.append(energy_distance_np(Xa, Xb))

    p = (np.sum(np.array(perm_ds) >= obs_d) + 1) / (len(perm_ds) + 1)
    dist_rows.append(
        {
            "cluster_name": cl,
            "edistance": obs_d,
            "pvalue": p,
            "n_WT": int(n_wt),
            "n_KO": int(n_ko),
        }
    )
    print(f"  {cl}: edist={obs_d:.4f}  perm_p={p:.4f}")

dist_df = pd.DataFrame(dist_rows)
_, padj_e, _, _ = multipletests(dist_df["pvalue"], method="fdr_bh")
dist_df["padj"] = padj_e
dist_df["significant"] = dist_df["padj"] < ALPHA
dist_df.to_csv("expression_shift_edistance.csv", index=False)

print(
    dist_df[["cluster_name", "edistance", "pvalue", "padj", "significant"]]
    .to_string(index=False)
)

sig_set_expr = set(dist_df[dist_df["significant"]]["cluster_name"])

# Per-cell z-scored distance to WT center
wt_center = X_pca_all[adata.obs[condition_key] == ref_group].mean(axis=0)
raw_dist = np.linalg.norm(X_pca_all - wt_center, axis=1)
adata.obs["edist_z"] = (raw_dist - raw_dist.mean()) / raw_dist.std()

# Colors per cluster
palette = sns.color_palette("tab20", len(cluster_order))
cl_color = {cl: palette[i] for i, cl in enumerate(cluster_order)}

tested = set(dist_df["cluster_name"])
ns_cls = [c for c in cluster_order if c in tested and c not in sig_set_expr]
sig_cls = [c for c in cluster_order if c in sig_set_expr]


def median_sort(cl_list, col="edist_z"):
    return sorted(
        cl_list,
        key=lambda c: adata.obs.loc[adata.obs["cluster_name"] == c, col].median(),
    )


plot_ord = median_sort(ns_cls) + median_sort(sig_cls)

fig, ax = plt.subplots(figsize=(16, 6), facecolor="white")
parts = ax.violinplot(
    [
        adata.obs.loc[adata.obs["cluster_name"] == cl, "edist_z"].values
        for cl in plot_ord
    ],
    positions=range(len(plot_ord)),
    widths=0.7,
    showmedians=False,
    showextrema=False,
    bw_method=0.3,
)

# Style violins
for body, cl in zip(parts["bodies"], plot_ord):
    body.set_facecolor(cl_color[cl])
    body.set_alpha(0.75)
    body.set_edgecolor("white")
    body.set_linewidth(0.5)

# Add medians and IQR
for i, cl in enumerate(plot_ord):
    vals = adata.obs.loc[
        adata.obs["cluster_name"] == cl, "edist_z"
    ].dropna()
    med = np.median(vals)
    q1 = np.percentile(vals, 25)
    q3 = np.percentile(vals, 75)
    ax.plot(
        [i],
        [med],
        marker="D",
        color="white",
        markersize=5,
        markeredgecolor="black",
        markeredgewidth=0.8,
        zorder=5,
    )
    ax.plot([i, i], [q1, q3], color="black", linewidth=1.5, zorder=4)

ax.axhline(0, color="black", linewidth=0.9)
if sig_cls:
    ax.axvline(len(ns_cls) - 0.5, color="black", linewidth=1, linestyle=":")

ax.set_xticks(range(len(plot_ord)))
ax.set_xticklabels(
    plot_ord, rotation=45, ha="right", fontsize=8, rotation_mode="anchor"
)
ax.set_ylabel("Expression shift value (edist z-score)", fontsize=10)
ax.set_title(
    "Expression shift per cluster — pertpy e-distance\n"
    "Non-significant left | Significant right of ⋮",
    fontsize=10,
    fontweight="bold",
)
remove_spines(ax)
plt.tight_layout()
plt.savefig("expression_shift_violin.png", dpi=300, bbox_inches="tight")
plt.show()
print("Plot 2 saved: expression_shift_violin.png")

## 11.5 Plot 3 – UMAP colored by LDA z-score

We fit a Linear Discriminant Analysis (LDA) model in PCA space to find the axis that best separates WT and KO cells:

- each cell gets an LDA score = position along the WT↔KO axis,
- we z-score this per-cell value (LDA z-score),
- positive = more KO-like, negative = more WT-like,
- UMAP is colored by LDA z-score in three panels: WT, KO, and all cells.

In [ ]:
# Binary labels for LDA (0 = WT, 1 = KO)
y_bin = (adata.obs[condition_key] == test_group).astype(int).values

lda = LinearDiscriminantAnalysis(n_components=1)
lda.fit(X_pca_all, y_bin)

lda_raw = lda.transform(X_pca_all).ravel()
lda_z = (lda_raw - lda_raw.mean()) / lda_raw.std()

# Orient axis so KO cells have positive mean
if lda_z[y_bin == 1].mean() < lda_z[y_bin == 0].mean():
    lda_z = -lda_z

adata.obs["lda_z"] = lda_z

# Per-cluster Wilcoxon test on LDA z-scores (KO vs WT)
wlx_rows = []
for cl in cluster_order:
    wt_v = adata.obs.loc[
        (adata.obs["cluster_name"] == cl)
        & (adata.obs[condition_key] == ref_group),
        "lda_z",
    ].values
    ko_v = adata.obs.loc[
        (adata.obs["cluster_name"] == cl)
        & (adata.obs[condition_key] == test_group),
        "lda_z",
    ].values
    if len(wt_v) < 5 or len(ko_v) < 5:
        continue
    _, p = stats.mannwhitneyu(ko_v, wt_v, alternative="two-sided")
    wlx_rows.append(
        {
            "cluster_name": cl,
            "pvalue": p,
            "median_WT": np.median(wt_v),
            "median_KO": np.median(ko_v),
        }
    )

wlx_df = pd.DataFrame(wlx_rows)
_, padj_w, _, _ = multipletests(wlx_df["pvalue"], method="fdr_bh")
wlx_df["padj"] = padj_w
wlx_df["significant"] = wlx_df["padj"] < ALPHA
wlx_df.to_csv("lda_wilcoxon.csv", index=False)
sig_set_lda = set(wlx_df[wlx_df["significant"]]["cluster_name"])

print(
    wlx_df[["cluster_name", "pvalue", "padj", "significant"]].to_string(index=False)
)

# UMAP panels
umap_xy = np.asarray(adata.obsm["X_umap"])
vmin, vmax = np.percentile(lda_z, 2), np.percentile(lda_z, 98)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor="white")

for ax, cond, title in zip(
    axes, [ref_group, test_group, None], [ref_group, test_group, "All cells"]
):
    if cond is not None:
        mask_c = (adata.obs[condition_key] == cond).values
        xy = umap_xy[mask_c]
        scores = lda_z[mask_c]
    else:
        xy = umap_xy
        scores = lda_z

    if cond is not None:
        bg = umap_xy[~mask_c]
        ax.scatter(
            bg[:, 0],
            bg[:, 1],
            c="#DDDDDD",
            s=3,
            linewidths=0,
            rasterized=True,
        )

    sc_plot = ax.scatter(
        xy[:, 0],
        xy[:, 1],
        c=scores,
        cmap="RdBu_r",
        vmin=vmin,
        vmax=vmax,
        s=5,
        linewidths=0,
        rasterized=True,
    )
    plt.colorbar(sc_plot, ax=ax, shrink=0.7, label="LDA z-score")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("UMAP 1", fontsize=9)
    ax.set_ylabel("UMAP 2", fontsize=9)
    ax.set_aspect("equal")
    remove_spines(ax)

fig.suptitle(
    "UMAP coloured by LDA z-score (cell loading on WT↔KO discriminant axis)\n"
    "Red = KO-like · Blue = WT-like · cluster 14 excluded",
    fontsize=11,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.savefig("umap_lda_zscore.png", dpi=300, bbox_inches="tight")
plt.show()

# Reference UMAP with cluster labels
fig, ax = plt.subplots(figsize=(8, 7), facecolor="white")
palette14 = sns.color_palette("tab20", len(cluster_order))
for cl, col in zip(cluster_order, palette14):
    mask_cl = (adata.obs["cluster_name"] == cl).values
    ax.scatter(
        umap_xy[mask_cl, 0],
        umap_xy[mask_cl, 1],
        c=[col],
        s=4,
        linewidths=0,
        label=cl,
        rasterized=True,
    )

ax.legend(
    markerscale=2,
    fontsize=7,
    frameon=False,
    loc="center left",
    bbox_to_anchor=(1, 0.5),
)
ax.set_title("UMAP — cluster annotation (cluster 14 excluded)", fontsize=11)
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
remove_spines(ax)
plt.tight_layout()
plt.savefig("umap_clusters.png", dpi=300, bbox_inches="tight")
plt.show()
print("Plot 3 saved: umap_lda_zscore.png, umap_clusters.png")

print("\nAll outputs written.")